# Phase 2 — `nuance`, on TriviaQA only

`plan_benzon.md` Part 1 / `plan_benzon_implementation.md` Phase 2, revised to match Phase 1's
final design (`notebooks_benzon/phase_0/1_commitment.ipynb`): the bespoke
`benzon:ontology_trivials` set is discarded here too — one dataset (TriviaQA, the paper's
own) for every sentiment, not a bespoke hand-curated one.

**Purpose.** `nuance` ("how nuanced and multi-sided is the answer above?") is graded by
`IntrinsicMetricThreshold` against `answer_set_entropy` — Shannon entropy of the model's
*own* next-token distribution when forced to answer in a single word: one forward pass' full
next-token softmax, not an empirical distribution over several resampled, decoded strings.
(An earlier "costly, faithful" design — sample the answer N times and bucket by normalized
exact match — was tried first and superseded: it passed on the ontology-trivials anchor
(rho=0.421) but collapsed to rho=0.073 on this notebook's own first TriviaQA run, because
paraphrases of the *same* fact, e.g. "J. G. Ballard" vs "James Graham Ballard", landed in
different buckets and inflated entropy on exactly the items the ground truth should call
low-entropy. See `vconf/metrics.py`'s `answer_set_entropy` docstring for the full account.)
TriviaQA carries no fixed difficulty tag the way the discarded ontology-trivials set did, so
the gate's "easier questions should show lower entropy than harder ones" check is reframed
against labeled correctness instead: an item the model actually gets right is, by construction, one
it knows — it should show lower next-token entropy than an item it gets wrong.

**Second surprise.** Fixing the bucketing artifact made entropy a *better* labeled correctness
predictor (correct/incorrect separation nearly doubled) but made self-report correlation
*worse* on TriviaQA (rho=0.073 → 0.003), while the same design stayed fine on
ontology-trivials (rho=0.421 → 0.301). That points at the self-report *question's wording*
rather than the ground truth as the remaining blocker.

**Follow-up A/B/C/D test, this notebook's actual focus below.** Four follow-up phrasings
compared side by side against the same entropy ground truth, same pattern as Phase 1's
`MACHINE_COMMITMENT`/`MACHINE_COMMITMENT_PARALLEL` A/B test: asking about the *count* of
plausible answers (`NUANCE_AMBIGUITY`) beats "nuance" as a stylistic property (`NUANCE`,
rho 0.221 vs. 0.003). Two more Likert variants (`NUANCE_RESAMPLE`, `NUANCE_ONE_WORD`) were
tried, didn't beat `NUANCE_AMBIGUITY`, and were removed — see `vconf/sentiment.py`'s comment
and git history for the wording and numbers. A non-Likert binary-confirmation mechanism was
also tried and removed by the same decision, despite actually beating every Likert variant on
TriviaQA (rho=0.391) — kept out for now for scope, not because it underperformed; see git
history if picking this line of investigation back up.

In [1]:
import json
import os
import pathlib
import sys

import matplotlib
import numpy as np
import pandas as pd

ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "vconf").is_dir())
sys.path.insert(0, str(ROOT))

HERE = pathlib.Path.cwd()
with open(HERE / "config.json") as f:
    MODEL_CONFIG = json.load(f)
MODEL_NAME = MODEL_CONFIG["model"]
os.environ["VCONF_MODEL"] = MODEL_NAME
CACHE_DIR = HERE / "cache" / MODEL_NAME
OUT_DIR = HERE / "out" / MODEL_NAME
FIGS_DIR = OUT_DIR / "figs"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
FIGS_DIR.mkdir(parents=True, exist_ok=True)
TRIAL_LOG_PATH = CACHE_DIR / "trial.json"
CONSTRUCT_PATH = CACHE_DIR / "construct.json"
SCALE = MODEL_CONFIG.get("scale", {})

from vconf import notebook as nb

_run_config = nb.run_config


def _run_config_with_overrides(base="gemma-categorical", **overrides):
    cfg = _run_config(base, **overrides)
    if MODEL_CONFIG.get("attn_implementation"):
        cfg = cfg.scaled(attn_implementation=MODEL_CONFIG["attn_implementation"])
    return cfg


nb.run_config = _run_config_with_overrides
from vconf import data as datamod
from vconf import ground_truth as GT
from vconf import metrics as M
from vconf import pipeline
from vconf.sentiment import NUANCE, NUANCE_AMBIGUITY, NUANCE_CERTAINTY, NUANCE_DEFINED, bands

cfg = nb.run_config("gemma-categorical", dataset="triviaqa", name="nuance-triviaqa")
print(nb.describe(cfg))

profile          : reduced
model            : Qwen/Qwen2.5-7B-Instruct (28 layers)
sentiment        : confidence  (ground truth: correctness)
prompt / dataset : categorical / triviaqa
layer sweep      : (0, 5, 11, 16, 22, 27)
trial counts     : {'steering': 24, 'patching': 24, 'noising': 32, 'swap': 24, 'attention': 24}
activation set   : 300   calibration set: 40
chat template    : True   attention impl: None
NOTE             : reduced profile — procedures, prompts, positions and
                   metrics follow the manual exactly, but the model and the
                   sample sizes are smaller than the paper's, so the numbers
                   here are not expected to match its reported values.


In [2]:
def heatmap(df, vmin=-1, vmax=1):
    """Diverging heatmap styling for a correlation table — blue=positive, red=negative,
    light gray for NaN (a self-report that collapsed to one class, not just weak)."""
    cmap = matplotlib.colormaps["coolwarm"].copy()
    cmap.set_bad(color="#f0f0f0")
    return df.style.format("{:.3f}", na_rep="—").background_gradient(cmap=cmap, vmin=vmin, vmax=vmax, axis=None)

## Dataset

n=100 TriviaQA questions — the same slice as
`notebooks_benzon/phase_0/1_commitment.ipynb` (the paper's own validation split, first
100 by dataset order), so this phase's items are directly comparable to Phase 1's rather than
drawn from a separate anchor set.

In [3]:
N = SCALE.get("triviaqa", 100)
items = datamod.load_dataset_items("triviaqa", limit=N)
print(f"{len(items)} TriviaQA questions")
pd.DataFrame([{"qid": i.qid, "question": i.question} for i in items]).head()

100 TriviaQA questions


,qid,question
0,tc_2,Who was the man behind The Chipmunks?
1,tc_33,Which Lloyd Webber musical premiered in the US...
2,tc_40,Who was the next British Prime Minister after ...
3,tc_49,Who had a 70s No 1 hit with Kiss You All Over?
4,tc_56,What claimed the life of singer Kathleen Ferrier?


In [4]:
loaded = nb.open_model(cfg, device_map=MODEL_CONFIG.get("device_map"))

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

## Cost note

`nuance`'s ground truth used to need `n_samples` *extra* generations per trial on top of the
shared Phase-0 pass — real, billable compute (`plan_benzon.md`'s cost note). The forced
single-word logit-entropy design replaces that with exactly one extra forward pass per trial
(no sampling loop), so there's no `n_samples`/cost tradeoff to size here anymore.

## Four follow-up-question variants — one shared Phase-0 pass

`NUANCE_AMBIGUITY` (count-of-plausible-answers framing) beats "nuance" as a stylistic
property (`NUANCE`, rho 0.221 vs. 0.003). `NUANCE_DEFINED` tests the baseline `NUANCE`
wording itself: same criterion, prefixed with an explicit dictionary definition of "nuance"
— is the original phrasing's poor showing because the bare word doesn't reliably evoke
"there could be multiple plausible answers" for the model, rather than a stylistic reading
unrelated to answer-identity uncertainty? All four compared side by side against the same
entropy ground truth in one `run_multi_sentiment` call (one shared Phase-0 pass, four
Phase-1 self-report passes). Correctness is graded separately via `nb.graded`, the same
`AliasCorrectness` grader Phase 1 uses.

In [5]:
VARIANTS = {
    "nuance": NUANCE,
    "nuance_ambiguity": NUANCE_AMBIGUITY,
    "nuance_certainty": NUANCE_CERTAINTY,
    "nuance_defined": NUANCE_DEFINED,
}

out = pipeline.run_multi_sentiment(loaded, items, cfg, VARIANTS)
by_qid_variant = {name: {t.qid: t for t in pipeline.filter_valid(trials)} for name, trials in out.items()}
for name, d in by_qid_variant.items():
    print(f"{len(d)}/{len(out[name])} {name} trials valid")

common_qids = sorted(set.intersection(*(set(d) for d in by_qid_variant.values())))
print(f"\n{len(common_qids)} qids valid under all four self-reports")

example_qid = common_qids[0]
example = by_qid_variant["nuance"][example_qid]
print("\nquestion :", example.question)
print("answer   :", example.answer)
for name, spec in VARIANTS.items():
    print(f"{name:20s}:", spec.classes[by_qid_variant[name][example_qid].class_index])

100/100 nuance trials valid
100/100 nuance_ambiguity trials valid
100/100 nuance_certainty trials valid
100/100 nuance_defined trials valid

100 qids valid under all four self-reports

question : Who had a 60s No 1 with Downtown?
answer   : The Mamas & the Papas
nuance              : Flat
nuance_ambiguity    : Nuanced
nuance_certainty    : Flat
nuance_defined      : Flat


## `nuance` ground truth — forced single-word logit entropy

`metrics.answer_set_entropy(trial, loaded, cfg)` doesn't fit
`IntrinsicMetricThreshold.metric_fn`'s bare `Callable[[Trial], float]` (it needs `loaded` to
run the forward pass and `cfg` to rebuild the right prompt) — `plan_benzon.md`'s wrinkle #2
resolves this by binding those at the point the run actually exists, not as a module-level
constant. `cfg` here is the *same* config `run_multi_sentiment` used for the shared Phase-0
pass (`sentiment=CONFIDENCE`) — reconstructing the prompt from anything else would ask a
different question than the one that actually produced this trial's answer. Entropy only
depends on the question and the shared Phase-0 config, not on which follow-up phrasing is
being tested, so it's computed once and reused across all three self-report variants.

Each trial's entropy is one forward pass over the phase-0 prompt plus a
"Answer with a single word." suffix, reading the full next-token softmax at that single
position — computed exactly once per trial, cached by `qid`, then wrapped in a cheap lookup
so `IntrinsicMetricThreshold.labels()` — which calls `metric_fn` once per trial internally to
find the pool median — doesn't trigger a second, redundant forward pass.

In [6]:
entropy_by_qid = {
    qid: M.answer_set_entropy(by_qid_variant["nuance"][qid], loaded, cfg)
    for qid in common_qids
}
raw_entropy = np.array([entropy_by_qid[q] for q in common_qids])
print(f"mean answer-set entropy: {raw_entropy.mean():.3f} nats  (0 = fully certain of one word)")

nuance_ground_truth = GT.IntrinsicMetricThreshold(name="answer_set_entropy", metric_fn=lambda t: entropy_by_qid[t.qid])
common_trials = [by_qid_variant["nuance"][q] for q in common_qids]
nuance_labels = nuance_ground_truth.labels(common_trials, common_trials)

mean answer-set entropy: 2.209 nats  (0 = fully certain of one word)


In [7]:
nb.graded(common_trials)
correct_mask = pipeline.correctness(common_trials).astype(bool)

entropy_df = pd.DataFrame({
    "qid": common_qids,
    "correct": correct_mask,
    "entropy": raw_entropy,
})
entropy_by_correctness = entropy_df.groupby("correct")["entropy"].agg(["mean", "count"])
display(entropy_by_correctness)

correct_mean_entropy = entropy_df.loc[entropy_df.correct, "entropy"].mean()
incorrect_mean_entropy = entropy_df.loc[~entropy_df.correct, "entropy"].mean()
print(f"correct mean entropy: {correct_mean_entropy:.3f}   incorrect mean entropy: {incorrect_mean_entropy:.3f}")

,mean,count
correct,,
False,2.780858,50
True,1.637580,50


correct mean entropy: 1.638   incorrect mean entropy: 2.781


In [8]:
rho_by_variant = {}
for name, spec in VARIANTS.items():
    selfreport = np.array([spec.class_midpoint[spec.classes[by_qid_variant[name][q].class_index]] for q in common_qids])
    rho_by_variant[name] = M.intrinsic_correlation(selfreport, raw_entropy)

rho_table = pd.Series(rho_by_variant).sort_values(ascending=False)
display(heatmap(rho_table.to_frame("rho")))

winner_name = rho_table.index[0]
winner_spec = VARIANTS[winner_name]
rho = rho_table.iloc[0]
print(f"winning follow-up variant: {winner_name} (rho={rho:.3f})")

distribution = M.class_histogram(
    np.array([by_qid_variant[winner_name][q].class_index for q in common_qids]), classes=winner_spec.classes
)
high_band, low_band = bands(cfg.model_key, winner_spec)
uses_high = sum(distribution[c] for c in high_band)
uses_low = sum(distribution[c] for c in low_band)
print("class distribution:", distribution)
print(f"{cfg.model_key}-adjusted high band ({high_band}): {uses_high}"
      f"   low band ({low_band}): {uses_low}")

,rho
nuance_ambiguity,0.221
nuance_certainty,0.148
nuance_defined,0.046
nuance,0.003


winning follow-up variant: nuance_ambiguity (rho=0.221)
class distribution: {'Flat': 19, 'Somewhat nuanced': 9, 'Nuanced': 72, 'Highly nuanced': 0}
qwen-adjusted high band (('Highly nuanced',)): 0   low band (('Flat',)): 19


## A second ground truth: binary {Yes, No} logit entropy — tested on TriviaQA first

`answer_set_entropy` forces a single-word answer and reads entropy over the *full*
vocabulary. `yes_no_logit_entropy` (defined below) instead reads the model's own {Yes, No}
logits directly at the Phase-0 answer position — no extra suffix needed, and no generation,
just a forward pass — built for genuinely binary-answer datasets like
`benzon:ontology_trivials` ("Is a dog an animal?"). TriviaQA's questions aren't yes/no, so
this is off-domain here: the model's actual continuation never starts with "Yes"/"No" at all,
it starts with a name/fact. Worth checking empirically rather than assuming — does this
produce a real (if noisy) signal, or is it degenerate here?

In [9]:
import torch

from vconf.models import final_logits_of_texts, render_prompt
from vconf.prompts import PHASE0_KIND, build_phase0_prompt


def yes_no_logit_entropy(trial, loaded, cfg) -> float:
    """Binary Shannon entropy (nats) of the model's own {Yes, No} logits at the
    Phase-0 answer position — on-target for questions whose true answer space
    is binary (e.g. `benzon:ontology_trivials`), off-domain for TriviaQA (its
    answers are never yes/no), tested here anyway rather than assumed.
    Reuses the exact Phase-0 prompt verbatim (no suffix needed) and reads
    logits instead of generating, same cost shape as `answer_set_entropy`."""
    kind = PHASE0_KIND[cfg.prompt_kind]
    built = build_phase0_prompt(trial.question, kind, sentiment=cfg.sentiment)
    rendered = render_prompt(loaded.tokenizer, built, cfg.use_chat_template)
    logits = final_logits_of_texts(loaded.model, loaded.tokenizer, [rendered.text])[0]
    yes_id = loaded.tokenizer(" Yes", add_special_tokens=False)["input_ids"][0]
    no_id = loaded.tokenizer(" No", add_special_tokens=False)["input_ids"][0]
    pair_logprobs = torch.log_softmax(logits[[yes_id, no_id]].float(), dim=-1)
    probs = pair_logprobs.exp()
    return float(-(probs * pair_logprobs).sum())


yesno_entropy_by_qid = {q: yes_no_logit_entropy(by_qid_variant["nuance"][q], loaded, cfg) for q in common_qids}
triviaqa_yesno_entropy = np.array([yesno_entropy_by_qid[q] for q in common_qids])
print(f"mean yes/no entropy on TriviaQA: {triviaqa_yesno_entropy.mean():.4f} nats  (max = ln(2) = {np.log(2):.3f})")
print(f"std: {triviaqa_yesno_entropy.std():.4f}   min: {triviaqa_yesno_entropy.min():.4f}   max: {triviaqa_yesno_entropy.max():.4f}")

triviaqa_yesno_rho = {}
for name, spec in VARIANTS.items():
    selfreport = np.array([spec.class_midpoint[spec.classes[by_qid_variant[name][q].class_index]] for q in common_qids])
    triviaqa_yesno_rho[name] = M.intrinsic_correlation(selfreport, triviaqa_yesno_entropy)

print("\nrho vs. yes/no ground truth on TriviaQA:")
display(heatmap(pd.Series(triviaqa_yesno_rho).sort_values(ascending=False).to_frame("rho")))

corr_grounds_triviaqa = M.intrinsic_correlation(triviaqa_yesno_entropy, raw_entropy)
print(f"Spearman rho(yes/no entropy, full-vocabulary entropy) on TriviaQA: {corr_grounds_triviaqa:.3f}")

mean yes/no entropy on TriviaQA: 0.3275 nats  (max = ln(2) = 0.693)
std: 0.2442   min: 0.0011   max: 0.6931

rho vs. yes/no ground truth on TriviaQA:


,rho
nuance_ambiguity,-0.127
nuance,-0.216
nuance_defined,-0.260
nuance_certainty,-0.285


Spearman rho(yes/no entropy, full-vocabulary entropy) on TriviaQA: -0.109


In [10]:
spot = pd.DataFrame({
    "question": [by_qid_variant["nuance"][q].question for q in common_qids],
    "answer": [by_qid_variant["nuance"][q].answer for q in common_qids],
    "entropy": raw_entropy,
    **{name: [spec.classes[by_qid_variant[name][q].class_index] for q in common_qids] for name, spec in VARIANTS.items()},
    "correct": [by_qid_variant["nuance"][q].correct for q in common_qids],
    "high_nuance (ground truth)": nuance_labels,
})
print("Highest answer-set entropy (least certain of a single word):")
display(spot.sort_values("entropy", ascending=False).head(5))
print("\nLowest answer-set entropy (most certain of a single word):")
display(spot.sort_values("entropy").head(5))

Highest answer-set entropy (least certain of a single word):


,question,answer,entropy,nuance,nuance_ambiguity,nuance_certainty,nuance_defined,correct,high_nuance (ground truth)
33,Who had an 80s No 1 hit with Hold On To The Ni...,Sheena Easton,5.207612,Nuanced,Nuanced,Flat,Flat,False,True
14,Which sitcom star appeared on the big screenin...,Steve Guttenberg,5.133749,Nuanced,Somewhat nuanced,Flat,Flat,False,True
6,"In music, who was Sweet and Innocent and Too Y...",This question seems to be referring to a chara...,4.817926,Somewhat nuanced,Nuanced,Nuanced,Nuanced,False,True
84,Who was the defending champion when Martina Na...,Chris Evert,4.794869,Somewhat nuanced,Nuanced,Nuanced,Flat,False,True
47,Who had a 70s No 1 hit with Kiss You All Over?,The question asks about a 70s No 1 hit with th...,4.760051,Flat,Somewhat nuanced,Flat,Flat,False,True



Lowest answer-set entropy (most certain of a single word):


,question,answer,entropy,nuance,nuance_ambiguity,nuance_certainty,nuance_defined,correct,high_nuance (ground truth)
53,Kagoshima international airport is in which co...,Japan,0.100908,Flat,Flat,Flat,Flat,True,False
64,Where did the Shinning Path terrorists operate?,The Shining Path (Sendero Luminoso) operated p...,0.129973,Nuanced,Nuanced,Flat,Somewhat nuanced,True,False
75,Which country does the airline TAAG come from?,TAAG is from Angola.,0.150395,Flat,Flat,Flat,Flat,True,False
34,Who directed the classic 30s western Stagecoach?,"The classic 1939 western ""Stagecoach"" was dire...",0.265280,Nuanced,Nuanced,Flat,Somewhat nuanced,True,False
54,In which sport could the Pacers take on the Pi...,Basketball,0.279920,Somewhat nuanced,Nuanced,Nuanced,Flat,True,False


## Phase 2 gate (`plan_benzon_implementation.md`, revised for the follow-up A/B/C test)

> the winning follow-up variant's self-report correlates with answer-set entropy (rho > 0.2)
> without collapsing to one class; correctly-answered items show lower entropy than
> incorrectly-answered ones. Same calibration-anchor logic as Phase 1's commitment A/B test
> — the wording comparison settles which follow-up question to keep, if any clears the bar.

A stop-and-fix gate, not a checkpoint to note and continue past
(`plan_benzon_implementation.md`'s "Running principle") — if this fails, the fix belongs
here, before Phase 3 (Synonyms) is built on top of an unvalidated `nuance`.

In [11]:
checks = {
    "correctly-answered items show lower entropy than incorrect ones (correct mean entropy < incorrect mean entropy)":
        correct_mean_entropy < incorrect_mean_entropy,
    f"winning variant ({winner_name}) self-report correlates with answer-set entropy (rho > 0.2)": rho > 0.2,
    f"winning variant ({winner_name}) self-report is not collapsed to a single class (top class < 90% of trials)":
        max(distribution.values()) / len(common_qids) < 0.9,
}
for name, ok in checks.items():
    print(f"{'PASS' if ok else 'FAIL'}  {name}")

gate_passed = all(checks.values())
print(f"\n{'GATE PASSED' if gate_passed else 'GATE FAILED'}"
      f" — {'proceed to Phase 3 (Synonyms)' if gate_passed else 'debug the prompt/extraction/entropy logic before building Phase 3 on top of this'}")

PASS  correctly-answered items show lower entropy than incorrect ones (correct mean entropy < incorrect mean entropy)
PASS  winning variant (nuance_ambiguity) self-report correlates with answer-set entropy (rho > 0.2)
PASS  winning variant (nuance_ambiguity) self-report is not collapsed to a single class (top class < 90% of trials)

GATE PASSED — proceed to Phase 3 (Synonyms)


**Interpretation.** If a follow-up variant clears the gate here, that's the wording to keep in
`sentiment.py` going forward — the losers can be dropped once this is confirmed stable,
mirroring Phase 1's `MACHINE_COMMITMENT`/`MACHINE_COMMITMENT_PARALLEL` resolution. Together
with Phase 1's commitment result (`notebooks_benzon/phase_0/1_commitment.ipynb`), this
is the second half of the calibration anchor `IntrinsicMetricThreshold` depends on everywhere
downstream — both now validated on the same TriviaQA slice instead of two different bespoke
sets. See the PASS/FAIL checklist above for whether it holds on this run — if it does,
`notebooks_benzon/phase_0/3_synonyms.ipynb` is next (`plan_benzon_implementation.md` Phase 3);
if it doesn't, the fix belongs here, not in Phase 3.

## Secondary check — all four follow-up variants on the ontology-trivials anchor

The `benzon:ontology_trivials` set is kept, not discarded (see `vconf/benzon_data.py`) — it
still carries a known easy/harder difficulty tag that TriviaQA doesn't. Worth checking
whether the same wording ranking holds on a completely different, difficulty-tiered dataset —
does `NUANCE_AMBIGUITY` stay on top here too, or was TriviaQA's ranking specific to that
dataset's real-world phrasing?

In [12]:
ontology_items = datamod.load_dataset_items("benzon:ontology_trivials", limit=SCALE.get("ontology_trivials"))
print(f"{len(ontology_items)} ontology-trivial questions")
display(pd.Series([item.meta["difficulty"] for item in ontology_items]).value_counts().to_frame("count"))

65 ontology-trivial questions


,count
harder,42
easy,23


In [13]:
cfg_ontology = nb.run_config(
    "gemma-categorical", dataset="benzon:ontology_trivials", name="nuance-ontology-trivials"
)
out_ontology = pipeline.run_multi_sentiment(loaded, ontology_items, cfg_ontology, VARIANTS)
ontology_by_qid_variant = {
    name: {t.qid: t for t in pipeline.filter_valid(trials)} for name, trials in out_ontology.items()
}
for name, d in ontology_by_qid_variant.items():
    print(f"{len(d)}/{len(out_ontology[name])} {name} trials valid")

ontology_common_qids = sorted(set.intersection(*(set(d) for d in ontology_by_qid_variant.values())))
print(f"\n{len(ontology_common_qids)} qids valid under all four self-reports")

65/65 nuance trials valid
65/65 nuance_ambiguity trials valid
65/65 nuance_certainty trials valid
65/65 nuance_defined trials valid

65 qids valid under all four self-reports


In [14]:
ontology_entropy_by_qid = {
    qid: M.answer_set_entropy(ontology_by_qid_variant["nuance"][qid], loaded, cfg_ontology)
    for qid in ontology_common_qids
}
ontology_raw_entropy = np.array([ontology_entropy_by_qid[q] for q in ontology_common_qids])

ontology_difficulty_by_qid = {item.qid: item.meta["difficulty"] for item in ontology_items}
ontology_entropy_df = pd.DataFrame({
    "qid": ontology_common_qids,
    "difficulty": [ontology_difficulty_by_qid[q] for q in ontology_common_qids],
    "entropy": ontology_raw_entropy,
})
display(ontology_entropy_df.groupby("difficulty")["entropy"].agg(["mean", "count"]))

ontology_easy_mean_entropy = ontology_entropy_df.loc[ontology_entropy_df.difficulty == "easy", "entropy"].mean()
ontology_harder_mean_entropy = ontology_entropy_df.loc[ontology_entropy_df.difficulty == "harder", "entropy"].mean()
print(f"easy mean entropy: {ontology_easy_mean_entropy:.3f}   harder mean entropy: {ontology_harder_mean_entropy:.3f}")

,mean,count
difficulty,,
easy,0.171792,23
harder,0.329475,42


easy mean entropy: 0.172   harder mean entropy: 0.329


In [15]:
ontology_rho_by_variant = {}
for name, spec in VARIANTS.items():
    selfreport = np.array(
        [spec.class_midpoint[spec.classes[ontology_by_qid_variant[name][q].class_index]] for q in ontology_common_qids]
    )
    ontology_rho_by_variant[name] = M.intrinsic_correlation(selfreport, ontology_raw_entropy)

ontology_rho_table = pd.Series(ontology_rho_by_variant).sort_values(ascending=False)
display(heatmap(ontology_rho_table.to_frame("rho")))

ontology_winner_name = ontology_rho_table.index[0]
ontology_winner_spec = VARIANTS[ontology_winner_name]
ontology_rho = ontology_rho_table.iloc[0]
print(f"winning follow-up variant on ontology-trivials: {ontology_winner_name} (rho={ontology_rho:.3f})")

ontology_distribution = M.class_histogram(
    np.array([ontology_by_qid_variant[ontology_winner_name][q].class_index for q in ontology_common_qids]),
    classes=ontology_winner_spec.classes,
)
print("class distribution:", ontology_distribution)

ontology_checks = {
    "easy items show lower entropy than harder ones (easy mean entropy < harder mean entropy)":
        ontology_easy_mean_entropy < ontology_harder_mean_entropy,
    f"winning variant ({ontology_winner_name}) self-report correlates with answer-set entropy (rho > 0.2)":
        ontology_rho > 0.2,
    f"winning variant ({ontology_winner_name}) self-report is not collapsed to a single class (top class < 90% of trials)":
        max(ontology_distribution.values()) / len(ontology_common_qids) < 0.9,
}
for name, ok in ontology_checks.items():
    print(f"{'PASS' if ok else 'FAIL'}  {name}")

ontology_gate_passed = all(ontology_checks.values())
print(f"\n{'PASS' if ontology_gate_passed else 'FAIL'} — nuance "
      f"{'also holds' if ontology_gate_passed else 'does not hold'} on the ontology-trivials anchor")

,rho
nuance_certainty,0.568
nuance_defined,0.434
nuance_ambiguity,0.310
nuance,0.301


winning follow-up variant on ontology-trivials: nuance_certainty (rho=0.568)
class distribution: {'Flat': 46, 'Somewhat nuanced': 0, 'Nuanced': 19, 'Highly nuanced': 0}
PASS  easy items show lower entropy than harder ones (easy mean entropy < harder mean entropy)
PASS  winning variant (nuance_certainty) self-report correlates with answer-set entropy (rho > 0.2)
PASS  winning variant (nuance_certainty) self-report is not collapsed to a single class (top class < 90% of trials)

PASS — nuance also holds on the ontology-trivials anchor


## New ground truth for yes/no-answerable datasets: binary {Yes, No} logit entropy

`answer_set_entropy` forces a single-word answer and reads entropy over the *full*
vocabulary — appropriate for TriviaQA, whose answer space genuinely isn't binary, but
wasteful for `benzon:ontology_trivials`, whose questions ("Is a dog an animal?") only ever
have two valid answers. Most of the full-vocabulary probability mass there is irrelevant:
words that were never a valid answer to a yes/no question in the first place. `yes_no_logit_entropy`
(defined above, tested there on off-domain TriviaQA first) reads the model's own {Yes, No}
logits directly at the Phase-0 answer position instead — on-target here, since this dataset's
questions genuinely are binary.

In [16]:
ontology_yesno_entropy_by_qid = {
    qid: yes_no_logit_entropy(ontology_by_qid_variant["nuance"][qid], loaded, cfg_ontology)
    for qid in ontology_common_qids
}
ontology_yesno_entropy = np.array([ontology_yesno_entropy_by_qid[q] for q in ontology_common_qids])
print(f"mean yes/no entropy: {ontology_yesno_entropy.mean():.3f} nats  (max = ln(2) = {np.log(2):.3f})")

yesno_rho_by_variant = {}
for name, spec in VARIANTS.items():
    selfreport = np.array(
        [spec.class_midpoint[spec.classes[ontology_by_qid_variant[name][q].class_index]] for q in ontology_common_qids]
    )
    yesno_rho_by_variant[name] = M.intrinsic_correlation(selfreport, ontology_yesno_entropy)

print("\nrho vs. NEW yes/no ground truth:")
display(heatmap(pd.Series(yesno_rho_by_variant).sort_values(ascending=False).to_frame("rho")))
print("(for comparison) rho vs. full-vocabulary answer_set_entropy:")
display(heatmap(ontology_rho_table.to_frame("rho")))

mean yes/no entropy: 0.031 nats  (max = ln(2) = 0.693)

rho vs. NEW yes/no ground truth:


,rho
nuance_ambiguity,0.441
nuance_certainty,0.402
nuance_defined,0.354
nuance,0.313


(for comparison) rho vs. full-vocabulary answer_set_entropy:


,rho
nuance_certainty,0.568
nuance_defined,0.434
nuance_ambiguity,0.310
nuance,0.301


In [17]:
yesno_entropy_df = pd.DataFrame({
    "qid": ontology_common_qids,
    "difficulty": [ontology_difficulty_by_qid[q] for q in ontology_common_qids],
    "yesno_entropy": ontology_yesno_entropy,
    "fullvocab_entropy": ontology_raw_entropy,
})
display(yesno_entropy_df.groupby("difficulty")["yesno_entropy"].agg(["mean", "count"]))

yesno_easy = yesno_entropy_df.loc[yesno_entropy_df.difficulty == "easy", "yesno_entropy"].mean()
yesno_harder = yesno_entropy_df.loc[yesno_entropy_df.difficulty == "harder", "yesno_entropy"].mean()
print(f"easy mean yes/no entropy: {yesno_easy:.3f}   harder mean yes/no entropy: {yesno_harder:.3f}")
print(f"{'PASS' if yesno_easy < yesno_harder else 'FAIL'}  easy items show lower yes/no entropy than harder ones")

corr_between_grounds = M.intrinsic_correlation(ontology_yesno_entropy, ontology_raw_entropy)
print(f"\nSpearman rho(yes/no entropy, full-vocabulary entropy): {corr_between_grounds:.3f}  "
      f"(agreement between the two ground truths)")

,mean,count
difficulty,,
easy,0.000103,23
harder,0.047403,42


easy mean yes/no entropy: 0.000   harder mean yes/no entropy: 0.047
PASS  easy items show lower yes/no entropy than harder ones

Spearman rho(yes/no entropy, full-vocabulary entropy): 0.359  (agreement between the two ground truths)


**Interpretation.** This secondary check corroborates (or flags a mismatch with) the
TriviaQA-based gate above using a completely different, difficulty-tiered dataset — since
`nuance`'s ground truth (`answer_set_entropy`) is dataset-agnostic by construction, agreement
between the two is a stronger signal than either alone. The TriviaQA gate above remains the
one that decides whether Phase 3 (Synonyms) proceeds, per Phase 1's revised design; a
mismatch here is worth a closer look, not an automatic blocker.

## Full matrix: every follow-up variant × {binary entropy, nonbinary logit} × {TriviaQA, ontology-trivials}

Everything above in one table — binary entropy = full-vocabulary `answer_set_entropy`, nonbinary logit = the
binary {Yes, No} logit-entropy ground truth (on-target for ontology-trivials, off-domain for
TriviaQA, included anyway for completeness).

In [18]:
full_matrix = pd.DataFrame({
    "TriviaQA x binary entropy": rho_by_variant,
    "TriviaQA x nonbinary logit": triviaqa_yesno_rho,
    "ontology x binary entropy": ontology_rho_by_variant,
    "ontology x nonbinary logit": yesno_rho_by_variant,
}).reindex(VARIANTS.keys())
full_matrix.index.name = "follow-up variant"

display(heatmap(full_matrix))
print(f"agreement between the two ground truths — TriviaQA: {corr_grounds_triviaqa:.3f}   "
      f"ontology: {corr_between_grounds:.3f}")

,TriviaQA x binary entropy,TriviaQA x nonbinary logit,ontology x binary entropy,ontology x nonbinary logit
follow-up variant,,,,
nuance,0.003,-0.216,0.301,0.313
nuance_ambiguity,0.221,-0.127,0.310,0.441
nuance_certainty,0.148,-0.285,0.568,0.402
nuance_defined,0.046,-0.260,0.434,0.354


agreement between the two ground truths — TriviaQA: -0.109   ontology: 0.359


## Log every cell to `trial.json`

Same shared calibration log as `1_commitment.ipynb` — `conclusion.ipynb` reads this back.

In [19]:
from vconf import trial_log as TL

records = []
for name in VARIANTS:
    records.append({"dataset": "triviaqa", "om": name, "gt": "entropy_gt", "rho": rho_by_variant[name], "n": len(common_qids)})
    records.append({"dataset": "triviaqa", "om": name, "gt": "nbgt", "rho": triviaqa_yesno_rho[name], "n": len(common_qids)})
    records.append({"dataset": "ontology_trivials", "om": name, "gt": "entropy_gt", "rho": ontology_rho_by_variant[name], "n": len(ontology_common_qids)})
    records.append({"dataset": "ontology_trivials", "om": name, "gt": "nbgt", "rho": yesno_rho_by_variant[name], "n": len(ontology_common_qids)})

TL.upsert(records, path=TRIAL_LOG_PATH)
print(f"logged {len(records)} cells to {TRIAL_LOG_PATH}")

logged 16 cells to /home/stud_homes/s7846062/fatass/home/thesis/experiment/code_morph/notebooks_benzon/phase_0_calibration/cache/qwen/trial.json
